# Building a LangChain Chain to Summarize Text

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langsmith.integrations.claude_agent_sdk import configure_claude_agent_sdk

from tavily import TavilyClient

load_dotenv()
configure_claude_agent_sdk()

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

Claude Agent SDK not installed.


In [31]:
from typing import Dict, Any

@tool(description="Search the web for information")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

In [32]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:nemotron3:33b", temperature=0.15)

agent = create_agent(
    model=model,
    tools=[web_search],
    checkpointer=InMemorySaver(),
)


In [33]:
from langchain.messages import HumanMessage

question = HumanMessage(content="""given the information about  Elon Reeve Musk, I want you to create:
    1. A short summary
    2. two interesting facts about them""")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [34]:
last_message = response['messages'][-1]
print(last_message.content)
print(json.dumps(last_message.usage_metadata, indent=4))
print(json.dumps(last_message.response_metadata, indent=4))

**Summary:**  
Elon Reeve Musk (born June 28, 1971) is a South African-born American entrepreneur and business magnate. He serves as CEO of Tesla and SpaceX, co-founded PayPal, and expanded into artificial intelligence with xAI. Musk became the world’s richest person in 2025, amassed a $744 billion net worth by mid-2026 (making him the first trillionaire), and briefly advised Donald Trump’s administration on government efficiency through America PAC.  

**Two Interesting Facts:**  
1. **Dual Citizenship at Birth**: Musk holds both South African and Canadian citizenships by birth, reflecting his parents’ origins (his mother was Canadian-born in Saskatchewan).  
2. **Trillionaire Milestone**: In June 2026, Forbes reported he became the first person to reach a net worth of $1 trillion, solidifying his status as the world’s wealthiest individual.
{
    "input_tokens": 2775,
    "output_tokens": 869,
    "total_tokens": 3644
}
{
    "model": "nemotron3:33b",
    "created_at": "2026-07-31T14